In [0]:
import pandas as pd
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

FEATURE_SCHEMA = {
    "vendor_name": "string",
    "passenger_count": "Int64",
    "trip_distance": "float64",
    "rate_code": "string",
    "payment_type": "string",
    "fare_amount": "float64",
    "tip_amount": "float64",
    "total_amount": "float64",
    "surcharge": "float64",
    "mta_tax": "float64",
    "tolls_amount": "float64",
    "start_lon": "float64",
    "start_lat": "float64",
    "end_lon": "float64",
    "end_lat": "float64",
    "pickup_hour": "Int64",
    "pickup_day_of_week": "Int64",
    "pickup_month": "Int64",
    "pickup_is_weekend": "Int64",
    "avg_speed_mph": "float64",
    "label": "float64",
    "is_rush_hour": "Int64",
    "is_night": "Int64",
    "haversine_distance": "float64",
    "manhattan_distance": "float64",
    "fare_per_mile": "float64",
    "tip_ratio": "float64",
    "tolls_ratio": "float64",
    "predicted_trip_duration": "float64"
}

STRING_COLS = [k for k, v in FEATURE_SCHEMA.items() if v == "string"]
INT_COLS = [k for k, v in FEATURE_SCHEMA.items() if v == "Int64"]
FLOAT_COLS = [k for k, v in FEATURE_SCHEMA.items() if v == "float64"]


def apply_feature_schema(df: pd.DataFrame, schema_map: dict = FEATURE_SCHEMA) -> pd.DataFrame:
    """
    Normalize pandas dtypes so that Spark does not infer VOID / NullType
    when a column happens to be fully null in the current batch.
    """
    df = df.copy()

    for col, dtype in schema_map.items():
        if col not in df.columns:
            continue

        if dtype == "string":
            df[col] = df[col].apply(lambda x: "unknown" if pd.isna(x) else str(x)).astype("string")

        elif dtype == "Int64":
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

        elif dtype == "float64":
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")

    return df


def build_output_spark_schema() -> StructType:
    """
    Explicit Spark schema for inference output table.
    This avoids Spark inferring VOID for fully-null columns.
    """
    return StructType([
        StructField("vendor_name", StringType(), True),
        StructField("passenger_count", IntegerType(), True),
        StructField("trip_distance", DoubleType(), True),
        StructField("rate_code", StringType(), True),
        StructField("payment_type", StringType(), True),
        StructField("fare_amount", DoubleType(), True),
        StructField("tip_amount", DoubleType(), True),
        StructField("total_amount", DoubleType(), True),
        StructField("surcharge", DoubleType(), True),
        StructField("mta_tax", DoubleType(), True),
        StructField("tolls_amount", DoubleType(), True),
        StructField("start_lon", DoubleType(), True),
        StructField("start_lat", DoubleType(), True),
        StructField("end_lon", DoubleType(), True),
        StructField("end_lat", DoubleType(), True),
        StructField("pickup_hour", IntegerType(), True),
        StructField("pickup_day_of_week", IntegerType(), True),
        StructField("pickup_month", IntegerType(), True),
        StructField("pickup_is_weekend", IntegerType(), True),
        StructField("avg_speed_mph", DoubleType(), True),
        StructField("label", DoubleType(), True),
        StructField("is_rush_hour", IntegerType(), True),
        StructField("is_night", IntegerType(), True),
        StructField("haversine_distance", DoubleType(), True),
        StructField("manhattan_distance", DoubleType(), True),
        StructField("fare_per_mile", DoubleType(), True),
        StructField("tip_ratio", DoubleType(), True),
        StructField("tolls_ratio", DoubleType(), True),
        StructField("predicted_trip_duration", DoubleType(), True),
    ])